In [ ]:
# ==============================================================================
# CELL 1: INSTALL LIBRARY, MOUNT DRIVE, & SETUP DIREKTORI
# ==============================================================================
!pip install kagglehub tf2onnx onnx opencv-python pandas numpy seaborn matplotlib scikit-learn tensorflow tqdm

import os
import shutil
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub
import tf2onnx
import onnx
from tqdm import tqdm
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as res_preprocess
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger

In [ ]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Setup Direktori Drive (Penyimpanan Permanen) & Lokal (Penyimpanan Cepat)
DRIVE_BASE_DIR = '/content/drive/MyDrive/Skripsi_EcoPlan'
DRIVE_DATASET_DIR = os.path.join(DRIVE_BASE_DIR, 'dataset')
DRIVE_MODEL_DIR = os.path.join(DRIVE_BASE_DIR, 'models')
LOCAL_DATASET_DIR = '/content/dataset_local'

os.makedirs(DRIVE_DATASET_DIR, exist_ok=True)
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

In [ ]:
# ==============================================================================
# CELL 2: MANAJEMEN DATASET (DOWNLOAD, COPY KE LOKAL) & SPLIT DATA
# ==============================================================================
# 1. Cek apakah dataset sudah ada di Drive. Jika belum, download via Kagglehub.
if len(os.listdir(DRIVE_DATASET_DIR)) == 0:
    print("\n[INFO] Dataset belum ada di Drive. Mengunduh dari Kaggle...")
    dataset_path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")

    # Cari folder utama yang berisi sub-folder kelas
    source_dataset = dataset_path
    for root, dirs, files in os.walk(dataset_path):
        if len(dirs) > 2:
            source_dataset = root
            break

    # Copy dari hasil download ke Google Drive agar tersimpan permanen
    print("\n[INFO] Menyalin dataset ke Google Drive...")
    for item in os.listdir(source_dataset):
        s = os.path.join(source_dataset, item)
        d = os.path.join(DRIVE_DATASET_DIR, item)
        if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
else:
    print("\n[INFO] Dataset sudah tersedia di Google Drive.")

# 2. Copy dataset dari Drive ke Lokal Colab untuk mempercepat proses Epoch I/O
if not os.path.exists(LOCAL_DATASET_DIR):
    print("\n[INFO] Menyalin dataset dari Drive ke Lokal Colab (Mempercepat Training)...")
    shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
else:
    print("\n[INFO] Dataset sudah ada di Lokal Colab.")

# 3. Labeling dan Split Dataset
data = []
for root, dirs, files in os.walk(LOCAL_DATASET_DIR):
    for img_file in files:
        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            folder_name = os.path.basename(root)
            data.append({'path': os.path.join(root, img_file), 'label': folder_name})

df_all = pd.DataFrame(data)
num_classes = len(df_all['label'].unique())

print("\n[INFO] Melakukan Pembagian Dataset (80:10:10)...")
train_df, temp_df = train_test_split(df_all, test_size=0.20, stratify=df_all['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)
print(f"Distribusi Split -> Training: {len(train_df)} | Validation: {len(val_df)} | Testing: {len(test_df)}")

In [ ]:
# ==============================================================================
# CELL 3: SETUP GENERATOR (AUGMENTASI & NORMALISASI)
# ==============================================================================
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

train_datagen = ImageDataGenerator(
    preprocessing_function=res_preprocess, rotation_range=20,
    width_shift_range=0.15, height_shift_range=0.15, shear_range=0.15,
    zoom_range=0.15, horizontal_flip=True, fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(preprocessing_function=res_preprocess)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col='path', y_col='label', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
val_gen = val_test_datagen.flow_from_dataframe(
    val_df, x_col='path', y_col='label', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col='path', y_col='label', target_size=IMG_SIZE,
    batch_size=1, class_mode='categorical', shuffle=False
)

In [ ]:
# ==============================================================================
# CELL 4: TRAINING (AUTO-RESUME & SAVE TO DRIVE)
# ==============================================================================
EPOCHS_LIMIT = 50
model_path_base = os.path.join(DRIVE_MODEL_DIR, "resnet50_best.keras")
csv_path = os.path.join(DRIVE_MODEL_DIR, "training_log.csv")

# Setup Callbacks yang langsung menyimpan ke Drive
callbacks_base = [
    EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.2, patience=3, min_lr=1e-6, monitor='val_loss'),
    ModelCheckpoint(filepath=model_path_base, save_best_only=True, monitor='val_loss', verbose=1),
    CSVLogger(csv_path, append=True)
]

# Cek Auto-Resume
last_epoch = 0
if os.path.exists(csv_path) and os.path.exists(model_path_base):
    try:
        df_log = pd.read_csv(csv_path)
        last_epoch = int(df_log['epoch'].iloc[-1]) + 1
        print(f"\n[INFO] Menemukan file checkpoint. Melanjutkan training dari epoch ke-{last_epoch}")
        model = load_model(model_path_base)
    except Exception as e:
        print(f"\n[INFO] Gagal resume: {e}. Membuat model baru...")
        last_epoch = 0

if last_epoch == 0:
    print("\n[INFO] Membangun Arsitektur ResNet50 Baru...")
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base_model.trainable = False

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy', metrics=['accuracy'])

# Mulai/Lanjutkan Training
if last_epoch < EPOCHS_LIMIT:
    print("\n[TRAINING] Memulai proses training...")
    model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS_LIMIT, initial_epoch=last_epoch,
        callbacks=callbacks_base
    )
else:
    print("\n[INFO] Training sudah mencapai batas maksimal epoch sebelumnya.")

In [ ]:
# ==============================================================================
# CELL 5: VISUALISASI GRAFIK ACCURACY & LOSS DARI FILE CSV
# ==============================================================================
if os.path.exists(csv_path):
    log_data = pd.read_csv(csv_path)

    plt.figure(figsize=(14, 5))

    # Grafik Akurasi
    plt.subplot(1, 2, 1)
    plt.plot(log_data['epoch'], log_data['accuracy'], label='Train Accuracy', linewidth=2)
    plt.plot(log_data['epoch'], log_data['val_accuracy'], label='Val Accuracy', linewidth=2)
    plt.title('Grafik Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True)

    # Grafik Loss
    plt.subplot(1, 2, 2)
    plt.plot(log_data['epoch'], log_data['loss'], label='Train Loss', linewidth=2)
    plt.plot(log_data['epoch'], log_data['val_loss'], label='Val Loss', linewidth=2)
    plt.title('Grafik Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================================================================
# CELL 6: EVALUASI TESTING & CONFUSION MATRIX
# ==============================================================================
print("\n[EVALUASI] Menghitung Metrik pada Data Testing...")
best_model = load_model(model_path_base)

# Prediksi
test_gen.reset()
preds = best_model.predict(test_gen, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes
class_labels = list(test_gen.class_indices.keys())

# Laporan Klasifikasi
print(f"\n-> Akurasi Testing: {accuracy_score(y_true, y_pred)*100:.2f}%\n")
print("[CLASSIFICATION REPORT]")
print(classification_report(y_true, y_pred, target_names=class_labels))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix')
plt.ylabel('Label Asli')
plt.xlabel('Label Prediksi')
plt.show()

In [ ]:
# ==============================================================================
# CELL 7: KONVERSI MODEL KE ONNX (SIMPAN DI DRIVE)
# ==============================================================================
import os

print("\n[INFO] Memulai proses konversi model ResNet50 ke format ONNX...")

# Pastikan variabel DRIVE_MODEL_DIR sudah didefinisikan sebelumnya
onnx_output_path = os.path.join(DRIVE_MODEL_DIR, "resnet50_garbage_ecoplan.onnx")
temp_saved_model_dir = os.path.join(DRIVE_MODEL_DIR, "temp_resnet50_saved_model")

# 1. Ekspor model Keras 3 ke format TensorFlow SavedModel
print("[INFO] Mengekspor model ke format SavedModel sementara...")
best_model.export(temp_saved_model_dir)

print("\n[INFO] Mengonversi SavedModel ke ONNX melalui CLI...")

In [ ]:
!python -m tf2onnx.convert --saved-model "{temp_saved_model_dir}" --output "{onnx_output_path}" --opset 13